# 第4章 深層学習の基礎 ― サンプルコード
『AI実装検定 A級・S級 完全版』の第4章のリストを、本文と同じ順に実行できます。

**使い方**：上のメニュー［ランタイム］→［すべてのセルを実行］。CPU のランタイムで動きます（GPU は不要）。

**本書の動作確認環境**（Python 3.13.17）：NumPy 2.5.3、scikit-learn 1.9.1、PyTorch 2.14.1、TensorFlow 2.21.0、Keras 3.15.1

In [ ]:
# 版をそろえたいとき（任意）：本書と同じ版を入れます。通常は Colab の標準版でそのまま動きます。
# !pip install -q numpy==2.5.3 scikit-learn==1.9.1 torch==2.14.1 tensorflow-cpu==2.21.0 keras==3.15.1
import sys; print(sys.version)


## リスト4.1　1 つのニューロンの計算

In [ ]:
import numpy as np

x = np.array([1.0, 2.0])
w = np.array([0.5, -0.25])
b = 0.1
z = w @ x + b
print(z)
print(max(0.0, z))                    # ReLU
print(round(1 / (1 + np.exp(-z)), 4)) # シグモイド
print(np.tanh(z).round(4))

**本書に載っている出力**
```
0.1
0.1
0.525
0.0997
```

## リスト4.2　活性化関数と softmax

In [ ]:
import numpy as np

z = np.array([-2.0, 0.0, 2.0])
relu = np.maximum(0, z)
sig = 1 / (1 + np.exp(-z))
print(relu)
print(sig.round(4))
print(np.tanh(z).round(4))
e = np.exp(z - z.max())               # 引いても結果は同じ（桁あふれ防止）
print((e / e.sum()).round(4), (e / e.sum()).sum().round(4))

**本書に載っている出力**
```
[0. 0. 2.]
[0.1192 0.5    0.8808]
[-0.964  0.     0.964]
[0.0159 0.1173 0.8668] 1.0
```

## リスト4.3　線形モデルは XOR を解けない

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([0, 1, 1, 0])
print(LogisticRegression().fit(X, y).score(X, y))    # 直線では分けられない

**本書に載っている出力**
```
0.5
```

## リスト4.4　2 入力・隠れ層 2・出力 1 の順伝播

In [ ]:
import numpy as np

# 2入力 -> 隠れ層2 (ReLU) -> 出力1
x = np.array([1.0, 2.0])
W1 = np.array([[1.0, -1.0], [0.5, 1.0]])
b1 = np.array([0.0, -1.0])
W2 = np.array([[2.0], [1.0]])
b2 = np.array([0.5])

h = np.maximum(0, x @ W1 + b1)
y = h @ W2 + b2
print(x @ W1 + b1, h)
print(y)
print(sum(W.size for W in [W1, b1, W2, b2]))   # パラメータ数

**本書に載っている出力**
```
[2. 0.] [2. 0.]
[4.5]
9
```

## リスト4.5　二乗和誤差と、勾配 y − t（数値微分での確認つき）

In [ ]:
import numpy as np

y = np.array([0.8, 0.2, 0.6])                      # 出力
t = np.array([1.0, 0.0, 1.0])                      # 正解
E = 0.5 * np.sum((y - t) ** 2)                     # 二乗和誤差
mse = np.mean((y - t) ** 2)                        # 平均二乗誤差
print(round(E, 4), round(mse, 4))
grad = y - t                                       # ∂E/∂y = y − t（1/2 が微分の 2 を打ち消す）
h = 1e-6
num = np.array([(0.5 * np.sum((y + h * e - t) ** 2) - 0.5 * np.sum((y - h * e - t) ** 2)) / (2 * h)
                for e in np.eye(3)])
print(grad.round(4), num.round(4))

**本書に載っている出力**
```
0.12 0.08
[-0.2  0.2 -0.4] [-0.2  0.2 -0.4]
```

## リスト4.6　softmax と交差エントロピー

In [ ]:
import torch
import torch.nn.functional as F

logits = torch.tensor([[2.0, 1.0, 0.0]])
target = torch.tensor([0])
p = F.softmax(logits, dim=1)
print(p.round(decimals=4))
print(round(F.cross_entropy(logits, target).item(), 4))
print(round(-torch.log(p[0, 0]).item(), 4))

**本書に載っている出力**
```
tensor([[0.6652, 0.2447, 0.0900]])
0.4076
0.4076
```

## リスト4.7　勾配は「softmax − ワンホット」

In [ ]:
import torch
import torch.nn.functional as F

logits = torch.tensor([[2.0, 1.0, 0.0]], requires_grad=True)
target = torch.tensor([0])
F.cross_entropy(logits, target).backward()
print(logits.grad.round(decimals=4))                 # softmax(z) - one-hot
print((F.softmax(logits, dim=1) - F.one_hot(target, 3)).detach().round(decimals=4))

**本書に載っている出力**
```
tensor([[-0.3348,  0.2447,  0.0900]])
tensor([[-0.3348,  0.2447,  0.0900]])
```

## リスト4.8　autograd による答え合わせ

In [ ]:
import torch

x = torch.tensor(1.0)
w1 = torch.tensor(2.0, requires_grad=True)
w2 = torch.tensor(-1.0, requires_grad=True)
h = torch.relu(w1 * x)      # h = 2
y = w2 * h                  # y = -2
loss = (y - 1) ** 2         # 9
loss.backward()
print(loss.item(), w1.grad.item(), w2.grad.item())

**本書に載っている出力**
```
9.0 6.0 -12.0
```

## リスト4.9　同じ関数 f(w)=w² を 20 回更新

In [ ]:
import torch

def run(opt_cls, **kw):
    w = torch.tensor([5.0], requires_grad=True)
    opt = opt_cls([w], **kw)
    for _ in range(20):
        opt.zero_grad()
        (w ** 2).sum().backward()          # f(w) = w^2, 最小値は w = 0
        opt.step()
    return round(w.item(), 4)

print("SGD lr=0.1        ", run(torch.optim.SGD, lr=0.1))
print("SGD momentum=0.9  ", run(torch.optim.SGD, lr=0.1, momentum=0.9))
print("Adam lr=0.1       ", run(torch.optim.Adam, lr=0.1))

**本書に載っている出力**
```
SGD lr=0.1         0.0576
SGD momentum=0.9   -1.7463
Adam lr=0.1        3.0603
```

## リスト4.10　RMSprop の 1 回の更新（手計算と torch.optim.RMSprop）

In [ ]:
import numpy as np
import torch

g, lr, beta, eps = 2.0, 0.1, 0.9, 1e-8
v = (1 - beta) * g * g                                 # 勾配の 2 乗の移動平均（初期値 0 から 1 回目）
step = lr * g / (np.sqrt(v) + eps)
print(round(v, 3), round(step, 4))

w = torch.tensor([1.0], requires_grad=True)
opt = torch.optim.RMSprop([w], lr=lr, alpha=beta, eps=eps)
(2.0 * w).sum().backward()                             # 勾配は常に 2.0
opt.step()
print(round(1.0 - w.item(), 4))                        # 1 回で動いた量

**本書に載っている出力**
```
0.4 0.3162
0.3162
```

## リスト4.11　重み減衰の有無で比較

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
Xtr = torch.linspace(-1, 1, 12).reshape(-1, 1)
ytr = Xtr ** 2 + 0.2 * torch.randn(12, 1)
Xte = torch.linspace(-1, 1, 100).reshape(-1, 1)
yte = Xte ** 2

def fit(wd, epochs=3000):
    torch.manual_seed(1)
    m = nn.Sequential(nn.Linear(1, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 1))
    opt = torch.optim.Adam(m.parameters(), lr=0.01, weight_decay=wd)
    for _ in range(epochs):
        opt.zero_grad()
        l = nn.functional.mse_loss(m(Xtr), ytr)
        l.backward()
        opt.step()
    with torch.no_grad():
        return nn.functional.mse_loss(m(Xtr), ytr).item(), nn.functional.mse_loss(m(Xte), yte).item()

for wd in [0.0, 0.01]:
    a, b = fit(wd)
    print("weight_decay", wd, "train", round(a, 4), "test", round(b, 4))

**本書に載っている出力**
```
weight_decay 0.0 train 0.0 test 0.0496
weight_decay 0.01 train 0.0401 test 0.017
```

## リスト4.12　ドロップアウトの学習時と評価時

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
d = nn.Dropout(p=0.5)
x = torch.ones(8)
d.train()
print(d(x))                     # 一部が 0、残りは 1/(1-p)=2 倍
d.eval()
print(d(x))                     # 評価時は何もしない

**本書に載っている出力**
```
tensor([0., 0., 2., 0., 0., 0., 2., 2.])
tensor([1., 1., 1., 1., 1., 1., 1., 1.])
```

## リスト4.13　バッチ正規化

In [ ]:
import torch
from torch import nn

x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
bn = nn.BatchNorm1d(1)
bn.train()
print(bn(x).detach().squeeze().round(decimals=4))
print(bn.running_mean)

**本書に載っている出力**
```
tensor([-1.3416, -0.4472,  0.4472,  1.3416])
tensor([0.2500])
```

## リスト4.14　テンソルの基本

In [ ]:
import torch

a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print(a.shape, a.dtype)
print(a @ a)
print(a.sum(dim=0), a.mean(dim=1))
print(a.reshape(4).shape, a.view(1, 4).shape)
print(torch.zeros(2, 3).shape)
n = a.numpy()
n[0, 0] = 99.0                    # NumPy 配列とメモリを共有
print(a[0, 0].item())

**本書に載っている出力**
```
torch.Size([2, 2]) torch.float32
tensor([[ 7., 10.],
        [15., 22.]])
tensor([4., 6.]) tensor([1.5000, 3.5000])
torch.Size([4]) torch.Size([1, 4])
torch.Size([2, 3])
99.0
```

## リスト4.15　自動微分と勾配の累積

In [ ]:
import torch

w = torch.tensor(2.0, requires_grad=True)
x = torch.tensor(3.0)
loss = (w * x - 5) ** 2           # (2*3 - 5)^2 = 1
loss.backward()
print(loss.item(), w.grad.item())  # d/dw = 2(wx-5)x = 2*1*3
w.grad.zero_()
loss2 = (w * x - 5) ** 2
loss2.backward()
print(w.grad.item())

# 勾配を累積してしまう例
v = torch.tensor(1.0, requires_grad=True)
for _ in range(3):
    (v * 5).backward()
print(v.grad.item())

**本書に載っている出力**
```
1.0 6.0
6.0
15.0
```

## リスト4.16　線形回帰の学習ループ

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
X = torch.linspace(-1, 1, 40).reshape(-1, 1)
y = 2 * X + 1                                  # 正解：y = 2x + 1
model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
opt = torch.optim.SGD(model.parameters(), lr=0.1)

for epoch in range(200):
    opt.zero_grad()                            # 勾配を 0 に
    loss = loss_fn(model(X), y)                # 順伝播と損失
    loss.backward()                            # 逆伝播
    opt.step()                                 # パラメータ更新
    if epoch % 50 == 0:
        print(epoch, round(loss.item(), 6))
print(round(model.weight.item(), 3), round(model.bias.item(), 3))

**本書に載っている出力**
```
0 1.627108
50 0.000987
100 1e-06
150 0.0
2.0 1.0
```

## リスト4.17　MLP は XOR を解ける

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y = torch.tensor([[0.], [1.], [1.], [0.]])
model = nn.Sequential(nn.Linear(2, 4), nn.Tanh(), nn.Linear(4, 1), nn.Sigmoid())
opt = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.BCELoss()
for _ in range(500):
    opt.zero_grad()
    loss = loss_fn(model(X), y)
    loss.backward()
    opt.step()
print((model(X) > 0.5).int().ravel().tolist())
print(sum(p.numel() for p in model.parameters()))

**本書に載っている出力**
```
[0, 1, 1, 0]
17
```

## リスト4.18　3 クラス分類と評価

In [ ]:
import torch
from torch import nn
from sklearn.datasets import make_blobs

torch.manual_seed(0)
Xn, yn = make_blobs(n_samples=300, centers=3, cluster_std=1.2, random_state=0)
X = torch.tensor(Xn, dtype=torch.float32)
y = torch.tensor(yn, dtype=torch.long)       # CrossEntropyLoss は整数のクラス番号
idx = torch.randperm(300)
tr, te = idx[:200], idx[200:]

model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 3))   # 出力は softmax 前の値
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
for epoch in range(100):
    model.train()
    opt.zero_grad()
    loss = loss_fn(model(X[tr]), y[tr])
    loss.backward()
    opt.step()
model.eval()
with torch.no_grad():                         # 評価時は勾配を計算しない
    acc = (model(X[te]).argmax(dim=1) == y[te]).float().mean()
print(round(loss.item(), 4))
print(round(acc.item(), 3))

**本書に載っている出力**
```
0.2669
0.86
```

## リスト4.19　TensorDataset と DataLoader

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X = torch.arange(10, dtype=torch.float32).reshape(10, 1)
y = torch.arange(10)
ds = TensorDataset(X, y)
dl = DataLoader(ds, batch_size=4, shuffle=False)
print(len(ds), len(dl))                      # データ数、1エポックのバッチ数（10 ÷ 4 を切り上げて 3、最後は 2 件）
for xb, yb in dl:
    print(xb.shape, yb.tolist())

**本書に載っている出力**
```
10 3
torch.Size([4, 1]) [0, 1, 2, 3]
torch.Size([4, 1]) [4, 5, 6, 7]
torch.Size([2, 1]) [8, 9]
```

## リスト4.20　Keras による 3 クラス分類（TensorFlow 2.21、Keras 3.15 で実行）

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"           # TensorFlow のログを減らす
import numpy as np
import keras

keras.utils.set_random_seed(0)
centers = np.array([[0, 0], [4, 0], [0, 4]], dtype="float32")     # 3 つの塊
y = np.repeat([0, 1, 2], 30)
X = centers[y] + np.random.default_rng(0).normal(0, 0.5, (90, 2)).astype("float32")

model = keras.Sequential([
    keras.layers.Input((2,)),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(3),
])
model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
model.fit(X, y, epochs=100, batch_size=32, verbose=0)   # 学習ループは fit が担当
loss, acc = model.evaluate(X, y, verbose=0)
print(model.count_params(), round(acc, 2))

**本書に載っている出力**
```
99 1.0
```

## リスト4.21　畳み込みを PyTorch で確かめる

In [ ]:
import torch
import torch.nn.functional as F

x = torch.tensor([[[[1., 2., 3.],
                    [4., 5., 6.],
                    [7., 8., 9.]]]])            # (N, C, H, W) = (1, 1, 3, 3)
w = torch.tensor([[[[1., 0.],
                    [0., 1.]]]])                # カーネル (出力ch, 入力ch, 2, 2)
print(F.conv2d(x, w).squeeze())
print(F.conv2d(x, w, bias=torch.tensor([1.0])).squeeze())   # バイアス b = 1

**本書に載っている出力**
```
tensor([[ 6.,  8.],
        [12., 14.]])
tensor([[ 7.,  9.],
        [13., 15.]])
```

## リスト4.22　公式と PyTorch の結果が一致するか

In [ ]:
import torch
import torch.nn.functional as F

rows = [(3, 2, 1, 0), (5, 3, 1, 0), (5, 3, 1, 1), (5, 3, 2, 0), (6, 3, 2, 0)]
for H, K, S, P in rows:
    formula = (H + 2 * P - K) // S + 1                  # 切り捨て割り算
    x = torch.zeros(1, 1, H, H)
    w = torch.zeros(1, 1, K, K)
    real = F.conv2d(x, w, stride=S, padding=P).shape[-1]
    print(H, K, S, P, "->", formula, real)

**本書に載っている出力**
```
3 2 1 0 -> 2 2
5 3 1 0 -> 3 3
5 3 1 1 -> 5 5
5 3 2 0 -> 2 2
6 3 2 0 -> 2 2
```

## リスト4.23　出力の形とパラメータ数

In [ ]:
import torch
from torch import nn

conv = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1)
x = torch.zeros(1, 3, 8, 8)                       # (N, C, H, W)
y = conv(x)
print(y.shape)                                    # 出力 (N, 16, 8, 8)
print(sum(p.numel() for p in conv.parameters()))  # 3*3*3*16 + 16
print(y.flatten(1).shape)                         # Linear に渡す前に平らにする
fc = nn.Linear(16 * 8 * 8, 10)
print(sum(p.numel() for p in fc.parameters()))    # 1024*10 + 10
print(1024 * 784, 5 * 5 + 1)                      # 全結合と畳み込みの重みの数

**本書に載っている出力**
```
torch.Size([1, 16, 8, 8])
448
torch.Size([1, 1024])
10250
802816 26
```

## リスト4.24　最大プーリングと、学習する重みの数

In [ ]:
import torch
from torch import nn

x = torch.tensor([[[[1., 3., 2., 4.],
                    [5., 6., 1., 2.],
                    [0., 2., 8., 7.],
                    [3., 1., 5., 9.]]]])
print(nn.MaxPool2d(2)(x).squeeze())
print(sum(p.numel() for p in nn.MaxPool2d(2).parameters()))   # 学習する重みの数

**本書に載っている出力**
```
tensor([[6., 4.],
        [3., 9.]])
0
```

## リスト4.25　手計算の表を NumPy で確かめる

In [ ]:
import numpy as np

def run(coef, xs):
    h, out = 0.0, []
    for x in xs:
        h = np.tanh(coef * h + x)          # h_t = tanh(coef * h_(t-1) + x_t)
        out.append(round(float(h), 3))
    return out

print(run(0.5, [1, 0, 0, 0]))              # 入力は最初の1回だけ
print(run(1.5, [1] + [0] * 19)[-3:], max(run(1.5, [1] + [0] * 19)))

**本書に載っている出力**
```
[0.762, 0.363, 0.18, 0.09]
[0.859, 0.859, 0.859] 0.859
```

## リスト4.26　同じ計算を nn.RNN で

In [ ]:
import torch
from torch import nn

rnn = nn.RNN(input_size=1, hidden_size=1, batch_first=True)   # 入力 (N, T, 特徴)
with torch.no_grad():
    rnn.weight_ih_l0.fill_(1.0)                # W_x
    rnn.weight_hh_l0.fill_(0.5)                # W_h
    rnn.bias_ih_l0.zero_()
    rnn.bias_hh_l0.zero_()
x = torch.tensor([[[1.0], [0.0], [0.0], [0.0]]])              # (1, 4, 1)
out, h_last = rnn(x)
print(out.squeeze().round(decimals=4))
print(out.shape, h_last.shape)

**本書に載っている出力**
```
tensor([0.7616, 0.3634, 0.1797, 0.0896], grad_fn=<RoundBackward1>)
torch.Size([1, 4, 1]) torch.Size([1, 1, 1])
```

## リスト4.27　10 時刻前への勾配の大きさ

In [ ]:
import torch

def grad_h0(w, steps=10):
    h0 = torch.tensor(0.3, requires_grad=True)
    h = h0
    for _ in range(steps):
        h = torch.tanh(w * h)                 # 入力は 0 にして記憶だけを見る
    h.backward()
    return h0.grad.item()

print("%.2e" % grad_h0(0.5))
print("%.2e" % grad_h0(1.0))
print("%.2e" % grad_h0(2.0))

**本書に載っている出力**
```
9.48e-04
4.90e-01
1.40e-06
```

## リスト4.28　RNN・GRU・LSTM のパラメータ数（入力 3、隠れ 4）

In [ ]:
from torch import nn

def n(m):
    return sum(p.numel() for p in m.parameters())

print(n(nn.RNN(3, 4)), n(nn.GRU(3, 4)), n(nn.LSTM(3, 4)))

**本書に載っている出力**
```
36 108 144
```